# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Original Author

Qingyue (Annie) Wang - Developer Advocate (Google)

[LinkedIn](https://www.linkedin.com/in/qingyuewang/)

[X](https://twitter.com/qingyuewang)

email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready to go!")

✅ All libraries are ready to go!


In [ ]:
# --- Securely Configure Your API Key ---

# Prompt the user for their API key securely
api_key = getpass('Enter your Google API Key: ')

# Get Your API Key HERE 👉 https://codelabs.developers.google.com/onramp/instructions#0
# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully! Let the fun begin.")

Enter your Google API Key: ··········
✅ API Key configured successfully! Let the fun begin.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!" # user prompt
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '8361cf29-c00d-4144-a5ef-c489c4e52d05'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable!

***

### **Relaxing & Artsy Day Trip: Sunnyvale & Santa Clara Serenity**

This itinerary combines free art appreciation, scenic walks, and delicious budget-friendly meals for a perfect spontaneous escape.

#### **Morning: Art Immersion in Santa Clara (10:30 AM - 12:30 PM)**

*   **Activity:** Begin your day with a visit to the **Triton Museum of Art** in Santa Clara. This museum offers free admission and free parking, focusing on contemporary and historical works by local Bay Area, regional, and national artists. It's a wonderful place to quietly wander through diverse exhibitions.
*   **Cost:** Free admission, free

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable!

***

### **Relaxing & Artsy Day Trip: Sunnyvale & Santa Clara Serenity**

This itinerary combines free art appreciation, scenic walks, and delicious budget-friendly meals for a perfect spontaneous escape.

#### **Morning: Art Immersion in Santa Clara (10:30 AM - 12:30 PM)**

*   **Activity:** Begin your day with a visit to the **Triton Museum of Art** in Santa Clara. This museum offers free admission and free parking, focusing on contemporary and historical works by local Bay Area, regional, and national artists. It's a wonderful place to quietly wander through diverse exhibitions.
*   **Cost:** Free admission, free parking.
*   **Operating Hours:** The museum is open Tuesday through Sunday from 11:00 AM to 4:30 PM.

#### **Lunch: Affordable Mediterranean Delights (12:45 PM - 1:45 PM)**

*   **Eatery:** Head to **The Falafel Stop** in Sunnyvale for a fresh and affordable lunch. This popular spot is known for its delicious falafel and shawarma wraps, with many items being budget-friendly.
*   **Cost:** ~$8-$15 per person.
*   **Operating Hours:** Open daily from 11:00 AM to 10:00 PM.

#### **Afternoon: Public Art & Nature Walk in Sunnyvale (2:00 PM - 5:30 PM)**

*   **Activity 1:** Embark on a **Self-Guided Public Art Tour in Sunnyvale**. The city boasts over 200 public art pieces, including the "Sun Flair" sculpture program where local artists transform sun sculptures displayed in various parks. You can find walking tour maps online, or simply explore a park like **Washington Park** to discover some of these vibrant installations.
*   **Cost:** Free.
*   **Activity 2:** Follow your art walk with a relaxing stroll at **Sunnyvale Baylands Park**. This park offers developed parkland, pathways, and access to the San Francisco Bay Trail, along with preserved wetlands. It's an ideal spot for birdwatching and enjoying the peaceful Bay Area scenery.
*   **Cost:** Vehicle entry fee is $6 from March through October. However, entry is free for pedestrians and bicyclists, making it very affordable if you walk or bike in.
*   **Operating Hours:** Baylands Park is open daily from 8:00 AM until 30 minutes after sunset.

#### **Evening: Casual Dinner & Downtown Stroll (6:00 PM - 8:00 PM)**

*   **Eatery:** Travel a short distance to Mountain View for dinner at **Taqueria La Espuela**. This no-frills taqueria is a local favorite, offering generous portions of authentic Mexican food at incredibly low prices, with many meals under $10.
*   **Cost:** ~$10-$20 per person.
*   **Operating Hours:** Open daily from 8:00 AM to 10:00 PM.
*   **Activity:** After dinner, enjoy a relaxed evening stroll through **Downtown Mountain View** along Castro Street. The area has a pleasant ambiance, often with street performers and interesting shops, perfect for unwinding.

Enjoy your affordable, relaxing, and artsy day trip!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '20ecd271-7a49-434c-bb05-3ea4ab81e73a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-78fc35fc-ed13-472f-aed5-ecc0c2a6250c',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xd8\x02\x01\xbe>\xf6\xfb\x04>\xed\xe4M\xbf\xcbH+\xce\xd0AQ\xe4\xb4\xe1{\xaa\x1a+\xe95\xf3\xf9q\xb2F;\xddSEO{\xdb\xa0\xbe\x91\x00\xf7\xb8%x\x1d\x95L\xe6\x80\xac\xa0\xef\x15\xba\rwd\x91=N\xd0K\xe9\xd3\xba\xf2fo\xca\xf9\xe9\xa4\x05\xb8j^u-\xeeq\xfc\x1f~\xb8\xf3q\xb89^B\x0c...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResp

The weather near Lake Tahoe is partly cloudy with a temperature of 38°F, and a west wind of 0 to 5 mph. It will be quite cool, so if you plan to go hiking, make sure to dress warmly and in layers!

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [ ]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [ ]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    # query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    # query = "Find the top-rated hotels in Bangkok from the database"
    query = "Find the top-rated hotels from the database."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels from the database.'

🚀 Running query for agent: 'trip_data_concierge' in session: '5fa96bf7-4e3d-4e59-a223-c9d12694d335'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels'
        },
        id='adk-1001b8f8-0955-4ad0-b781-88c147c8479f',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x8d\x02\x01\xbe>\xf6\xfb\xcc\x15D\xac\xad\x82\x0f\xa8\xce&\xed\xab\x8f\xf8#\x8a\xd0s{;\xea\x052Y\x06\x80\xcf\xa7\xb2\xdb[\xce\xd5\xa4{Br\xdf\xa6\xa3\xda\xc6E\xfce\x16\x84\xa0\xd5$\x0c\xad\xd3\xcfc\x10"Q\xa4\x80\xa0sL/Mdz\xfceq\x81\xd9l8\xc7Z\xf6\xc2\xd7\xf8\x93\xbb-\x1aU\xa0H\x9f\xa5...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMe

Based on the data you provided, I would recommend The Grand Hotel as the top-rated option. It boasts a 5-star rating, indicating a superior level of quality and service.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [ ]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [ ]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 66215120-728b-4d9f-8ee1-f6394470a5ca

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '66215120-728b-4d9f-8ee1-f6394470a5ca'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a memorable 2-day trip.

Let's start with **Day 1**:

Here's a possible itinerary:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & St. George's Castle**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Immerse yourself in its charming atmosphere and discover hidden viewpoints.
    *   Ascend to **St. George's Castle (Castelo de São Jorge)** for panoramic views of the city, t

Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a memorable 2-day trip.

Let's start with **Day 1**:

Here's a possible itinerary:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & St. George's Castle**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Immerse yourself in its charming atmosphere and discover hidden viewpoints.
    *   Ascend to **St. George's Castle (Castelo de São Jorge)** for panoramic views of the city, the Tagus River, and a glimpse into Lisbon's Moorish past.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca (tavern) in Alfama, perhaps trying some grilled sardines or a bacalhau dish.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral (Sé de Lisboa) & Baixa District**
    *   Visit the impressive **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing Romanesque and Gothic architecture.
    *   Stroll through the grid-patterned streets of the Baixa district, rebuilt after the 1755 earthquake, and admire the grandeur of Rossio Square and Praça do Comércio.
*   **Evening (7:00 PM onwards): Dinner & Fado Show in Mouraria or Alfama**
    *   Experience a true taste of Lisbon with dinner at a restaurant in the historic Mouraria or Alfama district, accompanied by a traditional live **Fado show**.

How does this sound for your first day?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '66215120-728b-4d9f-8ee1-f6394470a5ca'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""No problem at all! I can definitely suggest a historical alternative for your Day 1 morning that doesn't involve a castle.

Let's revise **Day 1**:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Immerse yourself in its charming atmosphere and discover hidden viewpoints.
    *   Instead of the castle, visit the **National Pantheon (Panteão Nacional)**. This majestic 17th-century church, with its impressive dome, is the resting place f

No problem at all! I can definitely suggest a historical alternative for your Day 1 morning that doesn't involve a castle.

Let's revise **Day 1**:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Immerse yourself in its charming atmosphere and discover hidden viewpoints.
    *   Instead of the castle, visit the **National Pantheon (Panteão Nacional)**. This majestic 17th-century church, with its impressive dome, is the resting place for many important Portuguese personalities and offers great views of the city and river from its terrace.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca (tavern) in Alfama, perhaps trying some grilled sardines or a bacalhau dish.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral (Sé de Lisboa) & Baixa District**
    *   Visit the impressive **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing Romanesque and Gothic architecture.
    *   Stroll through the grid-patterned streets of the Baixa district, rebuilt after the 1755 earthquake, and admire the grandeur of Rossio Square and Praça do Comércio.
*   **Evening (7:00 PM onwards): Dinner & Fado Show in Mouraria or Alfama**
    *   Experience a true taste of Lisbon with dinner at a restaurant in the historic Mouraria or Alfama district, accompanied by a traditional live **Fado show**.

How does this updated Day 1 itinerary sound to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '66215120-728b-4d9f-8ee1-f6394470a5ca'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's move on to an equally exciting **Day 2**, keeping your love for historic sites and great local food at the forefront.

Here's a proposed itinerary for Day 2:

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History & Iconic Pastries**
    *   Start your day by heading to the historic **Belém district**.
    *   Explore the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a stunning example of Manueline architecture, closely linked to Portugal's Age of Discoveries.
    *   Visit the iconic **Belém Tower (Torre de 

Excellent! I'm glad Day 1 is perfect. Let's move on to an equally exciting **Day 2**, keeping your love for historic sites and great local food at the forefront.

Here's a proposed itinerary for Day 2:

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History & Iconic Pastries**
    *   Start your day by heading to the historic **Belém district**.
    *   Explore the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a stunning example of Manueline architecture, closely linked to Portugal's Age of Discoveries.
    *   Visit the iconic **Belém Tower (Torre de Belém)** and the **Monument to the Discoveries (Padrão dos Descobrimentos)**, both celebrating Portugal's rich maritime past.
    *   No trip to Belém is complete without indulging in the world-famous **Pastéis de Belém** at the original factory, a true Lisbon culinary experience!
*   **Lunch (1:00 PM - 2:30 PM): Seafood Lunch in Belém**
    *   Enjoy a fresh seafood lunch at a local restaurant in the Belém area, perhaps by the riverfront, savoring the day's catch.
*   **Afternoon (2:30 PM - 6:00 PM): Carmo Convent Ruins & Santa Justa Lift**
    *   Travel back towards the city center.
    *   Explore the atmospheric ruins of the **Carmo Convent (Convento do Carmo)**, a Gothic church destroyed by the 1755 earthquake, which now houses an archaeological museum and provides a poignant historical perspective.
    *   Take a ride on the historic **Santa Justa Lift (Elevador de Santa Justa)** for panoramic views over the Baixa district and to connect to the charming Chiado neighborhood.
*   **Evening (7:00 PM onwards): Culinary Delights at Time Out Market & Bairro Alto Stroll**
    *   For dinner, immerse yourself in the vibrant atmosphere of the **Time Out Market (Mercado da Ribeira)**. This large food hall offers a fantastic array of Portuguese and international cuisine from various renowned chefs and vendors, allowing you to sample many different local specialties.
    *   After dinner, enjoy a leisurely stroll through the charming streets of **Bairro Alto**, known for its traditional architecture and lively evening ambiance.

How does this plan for your second day in Lisbon sound?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [ ]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: accbac3c-8422-4638-9147-78456aee86b9
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'accbac3c-8422-4638-9147-78456aee86b9'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! I'd be delighted to help you plan your 2-day trip to Lisbon, focusing on historic sites and delicious local food. Lisbon is a fantastic choice for both!

Let's start with **Day 1**:

Here's a proposed itinerary for your first day in Lisbon:

**Morning: Belém's Maritime History & Iconic Pastries**
*   **9:00 AM - Jerónimos Monastery (Mosteiro dos Jerónimos):** Start your day at this UNESCO World Heritage site, a magni

Hello! I'd be delighted to help you plan your 2-day trip to Lisbon, focusing on historic sites and delicious local food. Lisbon is a fantastic choice for both!

Let's start with **Day 1**:

Here's a proposed itinerary for your first day in Lisbon:

**Morning: Belém's Maritime History & Iconic Pastries**
*   **9:00 AM - Jerónimos Monastery (Mosteiro dos Jerónimos):** Start your day at this UNESCO World Heritage site, a magnificent monastery showcasing Manueline architecture. It's a true masterpiece of Portuguese Gothic style.
*   **11:00 AM - Belém Tower (Torre de Belém):** A short walk from the monastery, this iconic 16th-century fortress on the Tagus River is another UNESCO site and a symbol of Portugal's Age of Discovery.
*   **12:00 PM - Pastéis de Belém:** No visit to Belém is complete without tasting the original *Pastéis de Nata* (custard tarts) at the renowned Pastéis de Belém bakery. It's a national treasure!

**Lunch: Local Flavors at Time Out Market**
*   **1:30 PM - Time Out Market (Mercado da Ribeira):** Head back towards the city center and enjoy lunch at this famous food hall. It offers a wide variety of local restaurants and vendors, allowing you to sample different Portuguese dishes.

**Afternoon: Historic Alfama & Panoramic Views**
*   **3:00 PM - Alfama District & Lisbon Cathedral (Sé de Lisboa):** Explore Alfama, Lisbon's oldest and most charming district, with its winding cobblestone streets. Visit the Lisbon Cathedral, the city's oldest church, built in the 12th century in Romanesque style.
*   **4:30 PM - Miradouro das Portas do Sol & Miradouro de Santa Luzia:** Discover stunning panoramic views of Alfama and the Tagus River from these picturesque viewpoints.

**Evening: Castle Views & Traditional Dinner**
*   **6:00 PM - São Jorge Castle (Castelo de São Jorge):** Ascend to this historic castle, a former fortress with peacocks roaming the grounds, offering breathtaking views over the city, especially beautiful as the sun begins to set.
*   **8:00 PM - Dinner in Alfama or Baixa:** Enjoy traditional Portuguese dinner in one of the charming restaurants in the Alfama district or the more central Baixa area. Consider trying a *Bifana* (pork sandwich) for an authentic street food experience.

How does this sound for your first day? We can adjust anything you like!

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 54fba24f-39ef-4ca0-a4e7-7e4ec53294bb
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '54fba24f-39ef-4ca0-a4e7-7e4ec53294bb'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! I'm glad Day 1 looks good.

Let's plan for **Day 2**. To make sure I'm tailoring it perfectly, could you remind me of our destination, the overall trip duration, and any specific interests you have? I want to make sure Day 2 builds wonderfully on our previous plans!"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata() partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=68,
  prompt_token_count=314,
  prompt_t

Great! I'm glad Day 1 looks good.

Let's plan for **Day 2**. To make sure I'm tailoring it perfectly, could you remind me of our destination, the overall trip duration, and any specific interests you have? I want to make sure Day 2 builds wonderfully on our previous plans!

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
